In [ ]:
!pip install -q langchain langchain-community langchain-core faiss-cpu unstructured pypdf sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 54.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 7.3 MB/s eta 0:00:00
   

In [ ]:
!pip install -q langchain-text-splitters
!pip install -q langchain-openai
!pip install -q faiss-cpu
!pip install -q unstructured
!pip install -q pypdf
!pip install -q sentence-transformers
!pip install -q langchain-classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 44.8 MB/s eta 0:00:00


In [ ]:
# Import standard libraries for file handling and text processing
import os, pathlib, textwrap, glob

# Load documents from various sources
from langchain_community.document_loaders import UnstructuredURLLoader, TextLoader, PyPDFLoader

# Split long texts into smaller, manageable chunks for embedding
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Vector store to store and retrieve embeddings efficiently using FAISS
from langchain_community.vectorstores import FAISS

# Generate text embeddings using OpenAI or Hugging Face models
from langchain_openai import OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings, SentenceTransformerEmbeddings

# Use local LLMs for response generation
from langchain_community.llms import Ollama

# Build a retrieval chain that combines a retriever, a prompt, and an LLM
from langchain_classic.chains import ConversationalRetrievalChain

# Create prompts for the RAG system
from langchain_core.prompts import PromptTemplate

print("Libraries imported. You're good to go!")

Libraries imported. You're good to go!


In [ ]:
!pip show langchain

Name: langchain
Version: 1.3.13
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


In [ ]:
import langchain
print(langchain.__version__)

1.3.13


2. data preperation(injesting and chunking data)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import glob
pdf_paths = glob.glob("/content/drive/MyDrive/servicedata/*.pdf")

from langchain_community.document_loaders import PyPDFLoader
raw_docs = []
for pdf_path in pdf_paths:
    loader = PyPDFLoader(pdf_path)
    raw_docs.extend(loader.load())

print(f"Loaded {len(raw_docs)} PDF pages from {len(pdf_paths)} files.")

Loaded 8 PDF pages from 4 files.


chunk the text


In [ ]:
chunks = []
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = text_splitter.split_documents(raw_docs)

print(f"{len(chunks)} chunks ready for embedding")

42 chunks ready for embedding


building the retriever

1.loading the model



In [ ]:
embeddings = SentenceTransformerEmbeddings(model_name="thenlper/gte-small")

embedding_vector = embeddings.embed_query("Hello world!")
print(len(embedding_vector))

/tmp/ipykernel_433/4176108439.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name="thenlper/gte-small")


modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/68.1k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

384


2. building vector database


In [ ]:
vectordb = FAISS.from_documents(chunks, embeddings)
retreiever = vectordb.as_retriever(search_kwargs={"k": 8})
print(" vector store with", vectordb.index.ntotal, "embeddings")

 vector store with 42 embeddings


build the generation engine


so to build generation engine we have to install ollama and serve  gemma 3
1. install ollam
2. run it in background
3. pull gemma 3

In [ ]:
!apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!nvidia-smi

Sat Aug  1 17:43:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import subprocess
subprocess.Popen(["ollama", "serve"])

import time
time.sleep(5)

In [ ]:
!ollama pull gemma3:4b

testing the llm

In [ ]:
from langchain_community.llms import Ollama

llm = Ollama(model="gemma3:4b", temperature=0.1)

text = "hi, what is your name, do you know ai?"
response = llm.invoke(text)
print(response)

Hi there! My name is Gemma.

And yes, I absolutely know about AI! I *am* an AI – a large language model created by the Gemma team at Google DeepMind. It's pretty cool to be part of this field! 😊 

I can process and generate text, and I’m openly available for public use.


building the RAG
1. define system prompt
2. create a rag chain


define system prompt

In [ ]:
SYSTEM_TEMPLATE = """
You are a **Customer Support Chatbot**. Use only the information in CONTEXT to answer.
If the answer is not in CONTEXT, respond with “I'm not sure from the docs.”

Rules:
1) Use ONLY the provided <context> to answer.
2) If the answer is not in the context, say: "I don't know based on the retrieved documents."
3) Be concise and accurate. Prefer quoting key phrases from the context.
4) When possible, cite sources as [source: source] using the metadata.

CONTEXT:
{context}

USER:
{question}
"""

create a RAG chain

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_community.llms import Ollama

In [ ]:

!ollama pull nomic-embed-text

# 2. Create embeddings + vectorstore + retriever

In [ ]:

from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
prompt = PromptTemplate(input_variables=["context", "question"], template=SYSTEM_TEMPLATE)

llm = Ollama(model="gemma3:4b", temperature=0.1)

chain = ConversationalRetrievalChain.from_llm(
    llm,
    retriever,
    combine_docs_chain_kwargs={"prompt": prompt},
    return_source_documents=True,
)

In [ ]:
chain

ConversationalRetrievalChain(verbose=False, combine_docs_chain=StuffDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nYou are a **Customer Support Chatbot**. Use only the information in CONTEXT to answer.\nIf the answer is not in CONTEXT, respond with “I\'m not sure from the docs.”\n\nRules:\n1) Use ONLY the provided <context> to answer.\n2) If the answer is not in the context, say: "I don\'t know based on the retrieved documents."\n3) Be concise and accurate. Prefer quoting key phrases from the context.\n4) When possible, cite sources as [source: source] using the metadata.\n\nCONTEXT:\n{context}\n\nUSER:\n{question}\n'), llm=Ollama(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.13'}}, model='gemma3:4b', temperature=0.1), output_parser=StrOutputParser(), llm_kwargs={}), document_prompt=PromptTemplate(input_variables=['page_content'], in

validate the RAG

In [ ]:
test_questions = [
    "If I'm not happy with my purchase, what is your refund policy and how do I start a return?",
    "How long will delivery take for a standard order, and where can I track my package once it ships?",
    "What's the quickest way to contact your support team, and what are your operating hours?",
]

chat_history = []

for q in test_questions:
  result = chain({"question": q, "chat_history": chat_history})
  chat_history.append((q, result["answer"]))
  print(f"Q: {q}\nA: {result['answer']}\n")


Q: If I'm not happy with my purchase, what is your refund policy and how do I start a return?
A: If your gear doesn’t fit or just isn’t your vibe, send it back within **30 days** of delivery for a refund or free size exchange. [source: ROX-2025-05] To start a return, you will need to ensure the item is unworn and custom-embroidered items are not returned unless defective.

Q: How long will delivery take for a standard order, and where can I track my package once it ships?
A: A tracking link is emailed upon label creation. Status updates may take up to 12 h to appear after the parcel is scanned at origin terminal [source: 6].

Q: What's the quickest way to contact your support team, and what are your operating hours?
A: Live chat is available from 08:00–18:00 MT. You can also [contact logistics@everstorm.example](mailto:logistics@everstorm.example) or +1 (406) 555-0199. [source: PT]



connect it with streamlit.ui
